# 09 · Obstacles and Potential Fields

### Recap & why now
Two problems remain. The optimiser wanders off the straight-line route because nothing
forbids it, and there is a pillar in the room that nothing has mentioned.

This notebook handles both, with two genuinely different tools: a **corridor**, which is a
convex constraint the QP can take directly, and a **potential field**, which is not convex
and works by a completely different mechanism. Knowing when each applies is the point.

### Learning objectives
1. Add a **corridor** constraint and stop the trajectory wandering.
2. Explain why a **keep-out** region is not convex, and what that costs.
3. Build a **potential field** and watch it push a path around an obstacle.
4. Compare the two approaches honestly — neither is strictly better.
5. Verify clearance numerically rather than by looking at the plot.

In [ ]:
# === Standard setup used throughout this notebook ========================
import numpy as np                 # NumPy = fast vector/matrix math, so we never hand-write loops for arithmetic.
import matplotlib.pyplot as plt     # Matplotlib is our plotting engine for every static figure below.
from matplotlib import animation   # Turns a list of frames into a playable movie (used for the animations).
from mpl_toolkits.mplot3d import Axes3D   # Registers the '3d' projection used by the 3-D figures.
from IPython.display import HTML    # Embeds an animation as a self-contained JS player (no ffmpeg required).

%matplotlib inline
# Render animations as an in-browser JavaScript player so they always play, on any machine.
plt.rcParams["animation.html"] = "jshtml"
# Raise the embed size cap (MB) so longer clips are not silently cut off.
plt.rcParams["animation.embed_limit"] = 60
# One consistent, readable look for every figure in the manual.
plt.rcParams.update({"figure.dpi": 80, "font.size": 11, "axes.grid": True})
# Print numbers with 4 decimals and no scientific notation, so output is easy to eyeball.
np.set_printoptions(precision=4, suppress=True)
print("Setup complete — NumPy", np.__version__, "| Matplotlib", plt.matplotlib.__version__)

In [ ]:
# === Polynomial toolkit, built up over Notebooks 02-05 ===================

def poly_val(c, t, der=0):
    """Value of the polynomial c at time t, or of its `der`-th derivative."""
    out = 0.0
    for i in range(der, len(c)):                   # Terms below `der` differentiate away to zero.
        factor = 1.0
        for k in range(der):
            factor *= (i - k)                      # i(i-1)...(i-der+1), the falling factorial.
        out += c[i]*factor*t**(i - der)
    return out

def deriv_row(n, t, der):
    """Row r with r @ c = the der-th derivative at time t. One CONSTRAINT is one row."""
    r = np.zeros(n)
    for i in range(der, n):
        factor = 1.0
        for k in range(der):
            factor *= (i - k)
        r[i] = factor*t**(i - der)
    return r

def cost_matrix(n, T, der=4):
    """Q with c^T Q c = integral from 0 to T of (der-th derivative)^2 dt."""
    Q = np.zeros((n, n))
    for i in range(der, n):
        for j in range(der, n):
            ci = np.prod([i - k for k in range(der)])
            cj = np.prod([j - k for k in range(der)])
            power = i + j - 2*der + 1               # From integrating t^(i-der) * t^(j-der).
            Q[i, j] = ci*cj*T**power/power
    return Q

NCOEF = 8                                          # Order 7: eight coefficients, eight boundary conditions.
g = 9.81                                           # Gravity, needed whenever we turn acceleration into tilt.
print("polynomial toolkit ready — order %d, %d coefficients per segment per axis" % (NCOEF-1, NCOEF))

In [ ]:
# === The multi-segment solver from Notebook 06 ===========================

def seg_row(N, seg, t, der):
    """A row of the big constraint matrix that touches only segment `seg`."""
    r = np.zeros(N)
    r[seg*NCOEF:(seg+1)*NCOEF] = deriv_row(NCOEF, t, der)
    return r

def build_cost(times, der=4):
    """Block-diagonal Q: one cost_matrix per segment, stacked along the diagonal."""
    Q = np.zeros((len(times)*NCOEF, len(times)*NCOEF))
    for s, T in enumerate(times):
        Q[s*NCOEF:(s+1)*NCOEF, s*NCOEF:(s+1)*NCOEF] = cost_matrix(NCOEF, T, der)
    return Q

def build_constraints(waypoints, times):
    """Waypoints, rest at both ends, and continuity of velocity/acceleration/jerk at each join."""
    m = len(times); N = m*NCOEF
    rows, vals = [], []
    for s in range(m):                             # Every segment starts and ends on its waypoints.
        rows.append(seg_row(N, s, 0.0, 0));      vals.append(waypoints[s])
        rows.append(seg_row(N, s, times[s], 0)); vals.append(waypoints[s+1])
    for der in (1, 2, 3):                          # At rest, in every sense, at both ends.
        rows.append(seg_row(N, 0, 0.0, der));          vals.append(0.0)
        rows.append(seg_row(N, m-1, times[m-1], der)); vals.append(0.0)
    for s in range(m - 1):                         # The two sides of each join must AGREE...
        for der in (1, 2, 3):
            rows.append(seg_row(N, s, times[s], der) - seg_row(N, s+1, 0.0, der))
            vals.append(0.0)                       # ...but we never say WHAT they agree on.
    return np.array(rows), np.array(vals)

def solve_min_snap_1d(waypoints, times, der=4):
    """Equality-constrained QP, solved through the KKT system. One axis."""
    Q = build_cost(times, der)
    A, b = build_constraints(waypoints, times)
    KKT = np.block([[2*Q, A.T], [A, np.zeros((len(b), len(b)))]])
    sol = np.linalg.solve(KKT, np.concatenate([np.zeros(Q.shape[0]), b]))
    return sol[:Q.shape[0]].reshape(len(times), NCOEF)      # Drop the Lagrange multipliers.

def sample(coeffs, times, t, der=0):
    """Evaluate the piecewise polynomial at global time t."""
    edges = np.concatenate([[0.0], np.cumsum(times)])
    if t <= 0:         return poly_val(coeffs[0], 0.0, der)
    if t >= edges[-1]: return poly_val(coeffs[-1], times[-1], der)
    s = int(np.searchsorted(edges, t, side="right") - 1)
    return poly_val(coeffs[s], t - edges[s], der)

def min_snap_3d(waypoints, times):
    """Solve each axis separately and wrap the result in the ref(t) interface the cascade wants."""
    W = np.asarray(waypoints, float)
    coeffs = [solve_min_snap_1d(W[:, axis], times) for axis in range(3)]
    total = float(np.sum(times))
    def ref(t):
        t = min(max(t, 0.0), total)
        p = np.array([sample(coeffs[a], times, t, 0) for a in range(3)])
        v = np.array([sample(coeffs[a], times, t, 1) for a in range(3)])
        acc = np.array([sample(coeffs[a], times, t, 2) for a in range(3)])
        if t >= total:
            v = np.zeros(3); acc = np.zeros(3)     # Hold position once the trajectory is finished.
        return p, v, acc
    return ref, total, coeffs

ROUTE = np.array([(0, 0, 0), (0, 0, 1.5), (2.0, 0, 1.5), (2.0, 2.0, 1.5), (2.0, 2.0, 2.5), (0, 0, 1.5)])
DURATIONS = [2.5, 3.0, 3.0, 2.0, 4.0]
g = 9.81
print("solver ready — the standing route has %d waypoints and %d segments, %.1f s total" %
      (len(ROUTE), len(DURATIONS), sum(DURATIONS)))

## 1 · A corridor

The requirement in words: *at every instant, stay within $r$ metres of the straight line
between the waypoints you are travelling between.*

$$\left\|p(t) - \ell_s(t)\right\|_2 \le r$$

Notice what changed. Until now every constraint touched one axis, which is why Notebook 06
could solve three separate problems. A Euclidean norm mixes $x$, $y$ and $z$ into one
inequality, so the three axes must now be solved **together**. This is a second-order cone
constraint — still convex, no longer a plain QP — and CVXPY handles it in one line.

In [ ]:
import cvxpy as cp

def solve_corridor(route, times, tube=None, n_samples=20):
    """Minimum snap in 3-D with an optional tube around the straight-line route."""
    route = np.asarray(route, float)
    m = len(times); N = m*NCOEF
    Q = build_cost(times)
    C = [cp.Variable(N) for _ in range(3)]         # One coefficient vector per axis.
    constraints = []
    for axis in range(3):
        A_, b_ = build_constraints(route[:, axis], times)
        constraints.append(A_ @ C[axis] == b_)
    if tube is not None:
        for s in range(m):
            for t_ in np.linspace(0, times[s], n_samples):
                centre = route[s] + (route[s+1] - route[s])*(t_/times[s])   # On the straight line.
                offset = cp.hstack([seg_row(N, s, t_, 0) @ C[a_] - centre[a_] for a_ in range(3)])
                constraints.append(cp.norm(offset, 2) <= tube)              # The cone constraint.
    prob = cp.Problem(cp.Minimize(sum(cp.quad_form(C[a_], cp.psd_wrap(Q)) for a_ in range(3))),
                      constraints)
    for solver in (cp.CLARABEL, cp.SCS):
        try:
            prob.solve(solver=solver); break
        except cp.error.SolverError:
            continue
    else:
        return None, None, "solver_failed"
    if C[0].value is None:
        return None, None, prob.status
    return [cc.value.reshape(m, NCOEF) for cc in C], prob.value, prob.status

def route_deviation(coeffs, route, times, n=600):
    """How far the trajectory strays from the straight-line route, at its worst."""
    route = np.asarray(route, float)
    edges = np.concatenate([[0.0], np.cumsum(times)])
    worst = 0.0
    for t_ in np.linspace(0, edges[-1], n):
        s = min(int(np.searchsorted(edges, t_, side="right") - 1), len(times)-1)
        p_ = np.array([sample(coeffs[a_], times, t_) for a_ in range(3)])
        centre = route[s] + (route[s+1] - route[s])*((t_ - edges[s])/times[s])
        worst = max(worst, np.linalg.norm(p_ - centre))
    return worst

print("  tube      status      snap cost   worst deviation")
results = {}
for tube in (None, 1.0, 0.6):
    C_t, cost, status = solve_corridor(ROUTE, DURATIONS, tube=tube)
    results[tube] = C_t
    print("  %-9s %-11s %9.2f %14.3f m" %
          (str(tube), status, cost, route_deviation(C_t, ROUTE, DURATIONS)))
print("\nThe unconstrained trajectory strays almost a metre. A 0.6 m tube brings it into line,")
print("at more than double the snap cost — hugging the route means turning corners more sharply.")

## 2 · When more time does not help

Tighten the corridor and eventually no trajectory exists. The obvious response — give the
drone more time — is **wrong**, and Notebook 08 already explained why: scaling all the
times leaves the *path* exactly unchanged. A corridor is a constraint on the path.

What does help is changing the geometry: more waypoints. Subdivide each leg and the
polynomial has more places where it is pinned down.

In [ ]:
def subdivide(route, times, k=2):
    """Split every leg into k pieces, sharing its time equally."""
    route = np.asarray(route, float)
    new_route, new_times = [route[0]], []
    for s in range(len(times)):
        for i in range(1, k+1):
            new_route.append(route[s] + (route[s+1] - route[s])*i/k)
            new_times.append(times[s]/k)
    return np.array(new_route), new_times

print("  tightening the tube on the original 5-segment route:")
for tube in (0.6, 0.4, 0.25):
    _, _, status = solve_corridor(ROUTE, DURATIONS, tube=tube)
    print("    tube %.2f m -> %s" % (tube, status))

print("\n  does more TIME help? (Notebook 08 says it cannot)")
for scale in (1.0, 2.0, 4.0):
    _, _, status = solve_corridor(ROUTE, [t_*scale for t_ in DURATIONS], tube=0.25)
    print("    times x%.1f (total %5.1f s) -> %s" % (scale, scale*sum(DURATIONS), status))

print("\n  does more GEOMETRY help? (same total time, more waypoints)")
for k in (1, 2, 3):
    r_k, t_k = subdivide(ROUTE, DURATIONS, k) if k > 1 else (ROUTE, DURATIONS)
    for tube in (0.4, 0.25):
        _, cost, status = solve_corridor(r_k, t_k, tube=tube)
        print("    k=%d (%2d segments), tube %.2f -> %-12s %s" %
              (k, len(t_k), tube, status, "" if cost is None else "cost %.0f" % cost))
print("\nSubdivision works and time does not, exactly as the scaling law predicts.")

## 3 · The obstacle, and why it is different

Now the request everybody wants: *do not fly through that pillar.*

$$\left\|p(t) - c_{\text{obs}}\right\|_2 \ge r$$

Look at the inequality sign. Every constraint so far said "stay **inside** a convex set".
This one says "stay **outside**", and the outside of a ball is not convex — take two points
on opposite sides and the line between them goes straight through the middle.

One non-convex constraint destroys the guarantee that made this whole project reliable.
Sections 4 and 5 show the two standard responses.

In [ ]:
OBSTACLE = np.array([2.0, 1.0, 1.5])               # A pillar on the third leg.
OBS_RADIUS = 0.45

def clearance(coeffs, times, centre, n=800):
    """Closest approach of the trajectory to a point."""
    grid = np.linspace(0, sum(times), n)
    return min(np.linalg.norm(np.array([sample(coeffs[a_], times, t_) for a_ in range(3)]) - centre)
               for t_ in grid)

d_now = clearance(results[0.6], DURATIONS, OBSTACLE)
print("the corridor-constrained trajectory passes within %.3f m of the pillar's centre." % d_now)
print("the pillar has radius %.2f m -> %s\n" % (OBS_RADIUS, "COLLISION" if d_now < OBS_RADIUS else "clear"))

a_pt, b_pt = np.array([1.0, 1.0, 1.5]), np.array([3.0, 1.0, 1.5])   # Two points either side.
mid = 0.5*(a_pt + b_pt)
print("two points either side of the pillar:")
print("   A is %.2f m from the centre — outside ✔" % np.linalg.norm(a_pt - OBSTACLE))
print("   B is %.2f m from the centre — outside ✔" % np.linalg.norm(b_pt - OBSTACLE))
print("   their midpoint is %.2f m — INSIDE ✘" % np.linalg.norm(mid - OBSTACLE))
print("\nA set where the segment joining two members leaves the set is by definition not convex.")
print("Every guarantee we have relied on — unique optimum, no local minima, a trustworthy")
print("'infeasible' — comes from convexity, and this constraint does not have it.")

## 4 · Response one: route around, then optimise

The most common answer in practice. A discrete planner finds a collision-free sequence of
waypoints; the QP then smooths within a corridor that is known to be clear.

The honest framing: **the QP does not avoid obstacles.** It produces a smooth trajectory
inside a region that something else has already certified as safe. Here we place the
detour waypoint by hand.

In [ ]:
DETOUR = np.array([(0, 0, 0), (0, 0, 1.5), (2.0, 0, 1.5),
                   (2.9, 1.0, 1.5),                # Pushed out around the pillar.
                   (2.0, 2.0, 1.5), (2.0, 2.0, 2.5), (0, 0, 1.5)])
DETOUR_TIMES = [2.5, 3.0, 1.8, 1.8, 2.0, 4.0]
C_avoid, cost_avoid, status = solve_corridor(DETOUR, DETOUR_TIMES, tube=0.5)
d_avoid = clearance(C_avoid, DETOUR_TIMES, OBSTACLE)
print("detour route: %s, snap cost %.1f" % (status, cost_avoid))
print("closest approach %.3f m against a radius of %.2f -> %s" %
      (d_avoid, OBS_RADIUS, "clear ✔" if d_avoid > OBS_RADIUS else "COLLISION"))

fig = plt.figure(figsize=(11, 4.2))
for k, (C_k, route_k, times_k, title) in enumerate(
        [(results[0.6], ROUTE, DURATIONS, "original route"),
         (C_avoid, DETOUR, DETOUR_TIMES, "with a detour waypoint")]):
    ax = fig.add_subplot(1, 2, k+1, projection="3d")
    grid = np.linspace(0, sum(times_k), 500)
    P = np.array([[sample(C_k[a_], times_k, t_) for a_ in range(3)] for t_ in grid])
    ax.plot(P[:, 0], P[:, 1], P[:, 2], color="C0", lw=2.2)
    ax.plot(route_k[:, 0], route_k[:, 1], route_k[:, 2], "*", color="k", ms=9)
    u, v = np.mgrid[0:2*np.pi:24j, 0:np.pi:12j]
    ax.plot_surface(OBSTACLE[0] + OBS_RADIUS*np.cos(u)*np.sin(v),
                    OBSTACLE[1] + OBS_RADIUS*np.sin(u)*np.sin(v),
                    OBSTACLE[2] + OBS_RADIUS*np.cos(v), color="C3", alpha=0.35)
    ax.set_title(title, fontsize=10); ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
    ax.view_init(elev=20, azim=-70)
plt.tight_layout(); plt.show()

## 5 · Response two: a potential field

A completely different mechanism, and the one on the project's reading list. Give the goal
an **attractive** potential and every obstacle a **repulsive** one, then follow the
negative gradient downhill:

$$U(p) = \underbrace{\tfrac{1}{2}k_a\|p - p_{\text{goal}}\|^2}_{\text{pull}}
+ \underbrace{\tfrac{1}{2}k_r\left(\tfrac{1}{d} - \tfrac{1}{d_0}\right)^2}_{\text{push, within } d_0}$$

No optimiser, no constraints, no matrices — just a vector field you can evaluate anywhere.
It is fast, it works online, and it has a well-known weakness that the next cell finds.

In [ ]:
def potential_gradient(p, goal, obstacles, k_att=1.0, k_rep=0.6, d0=1.0):
    """Negative gradient of the potential: the direction to move."""
    force = -k_att*(np.asarray(p, float) - goal)   # Attraction, proportional to the distance.
    for centre, radius in obstacles:
        diff = np.asarray(p, float) - centre
        d = max(np.linalg.norm(diff) - radius, 1e-3)     # Distance to the SURFACE, not the centre.
        if d < d0:
            force += k_rep*(1/d - 1/d0)*(1/d**2)*(diff/np.linalg.norm(diff))
    return force

def follow_field(start, goal, obstacles, step=0.02, n=1200):
    """Walk downhill from start toward goal, avoiding obstacles."""
    p_ = np.asarray(start, float).copy(); path = [p_.copy()]
    for _ in range(n):
        f = potential_gradient(p_, goal, obstacles)
        norm = np.linalg.norm(f)
        if norm < 1e-6: break
        p_ = p_ + step*f/max(norm, 1.0)            # Normalise long steps, so it does not overshoot.
        path.append(p_.copy())
        if np.linalg.norm(p_ - goal) < 0.05: break
    return np.array(path)

obstacles = [(OBSTACLE, OBS_RADIUS)]
path = follow_field([2.0, -0.5, 1.5], [2.0, 2.5, 1.5], obstacles)
print("potential-field path: %d steps, closest approach to the pillar %.3f m (radius %.2f) -> %s" %
      (len(path), min(np.linalg.norm(p_ - OBSTACLE) for p_ in path), OBS_RADIUS,
       "clear ✔" if min(np.linalg.norm(p_ - OBSTACLE) for p_ in path) > OBS_RADIUS else "COLLISION"))

trapped = follow_field([2.0, -0.5, 1.5], [2.0, 2.5, 1.5], obstacles, n=1200)
reached = np.linalg.norm(trapped[-1] - np.array([2.0, 2.5, 1.5])) < 0.1
print("did it reach the goal? %s" % ("yes" if reached else "NO — stuck in a local minimum"))

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.0, 4.0))
xs, ys = np.meshgrid(np.linspace(0.5, 3.5, 30), np.linspace(-1.0, 3.0, 30))
U = np.zeros_like(xs)
for i in range(xs.shape[0]):
    for j in range(xs.shape[1]):
        p_ = np.array([xs[i, j], ys[i, j], 1.5])
        d = max(np.linalg.norm(p_ - OBSTACLE) - OBS_RADIUS, 1e-3)
        U[i, j] = 0.5*np.linalg.norm(p_ - np.array([2.0, 2.5, 1.5]))**2
        if d < 1.0:
            U[i, j] += 0.5*0.6*(1/d - 1/1.0)**2
a1.contourf(xs, ys, np.clip(U, 0, 8), levels=25, cmap="viridis")
a1.plot(path[:, 0], path[:, 1], color="C3", lw=2.4, label="path downhill")
circ = plt.Circle((OBSTACLE[0], OBSTACLE[1]), OBS_RADIUS, color="white", alpha=0.8)
a1.add_patch(circ)
a1.plot(2.0, 2.5, "*", color="C1", ms=15)
a1.set_xlabel("x [m]"); a1.set_ylabel("y [m]"); a1.legend(fontsize=8); a1.set_aspect("equal")
a1.set_title("The potential landscape, and the walk down it")

a2.plot(np.linalg.norm(path - OBSTACLE, axis=1), color="C0", lw=2)
a2.axhline(OBS_RADIUS, color="C3", ls="--", lw=1.5)
a2.text(len(path)*0.5, OBS_RADIUS+0.06, "obstacle surface", fontsize=8, color="C3")
a2.set_xlabel("step"); a2.set_ylabel("distance to the pillar centre [m]")
a2.set_title("Clearance, checked rather than eyeballed")
plt.tight_layout(); plt.show()

## 🧪 Try it yourself

**E1.** Compare the corridor and the potential field honestly. What does each guarantee,
and what does each fail at?

**E2.** Place the obstacle **directly** between the start and the goal and see what the
potential field does. This is the method's famous weakness.

In [ ]:
# --- Solution E1 ---
print("E1: corridor — convex, so the solver returns the unique best answer or an honest")
print("    'infeasible'. It optimises the whole trajectory at once and respects the vehicle's")
print("    smoothness. But it cannot express 'stay outside', so something else must first find a")
print("    safe route for it to smooth.")
print("    potential field — handles keep-out directly, runs online with no solver, and reacts to")
print("    obstacles that appear after planning. But it optimises nothing, gives no smoothness")
print("    guarantee, and can get stuck. E2 shows how.")
print("\n    In practice they are used TOGETHER: a field or a discrete planner finds a rough safe")
print("    route, and a QP smooths it inside a corridor known to be clear — which is exactly what")
print("    Section 4 did by hand.")

# --- Solution E2 ---
start, goal = np.array([0.5, 1.0, 1.5]), np.array([3.5, 1.0, 1.5])   # Pillar exactly between them.
stuck = follow_field(start, goal, obstacles, n=1500)
final_gap = np.linalg.norm(stuck[-1] - goal)
print("\nE2: obstacle placed exactly on the line from start to goal.")
print("    the walk stopped %.3f m from the goal after %d steps -> %s" %
      (final_gap, len(stuck), "arrived" if final_gap < 0.1 else "STUCK"))
print("    Directly behind the obstacle the repulsive push and the attractive pull are exactly")
print("    opposite, so their sum is zero: a local minimum that is not the goal. Nothing in the")
print("    method can detect it, because 'downhill' is all it knows.")
print("    Standard fixes are all heuristics — add a random perturbation, add a rotational term,")
print("    or fall back to a discrete planner. That is the honest cost of leaving convexity:")
print("    the method becomes fast and local, and stops being able to promise anything.")

fig, ax = plt.subplots(figsize=(6.2, 3.6))
ax.contourf(xs, ys, np.clip(U, 0, 8), levels=20, cmap="viridis", alpha=0.6)
ax.plot(stuck[:, 0], stuck[:, 1], color="C3", lw=2.4)
ax.plot(*start[:2], "o", color="C0", ms=9); ax.plot(*goal[:2], "*", color="C1", ms=15)
ax.add_patch(plt.Circle((OBSTACLE[0], OBSTACLE[1]), OBS_RADIUS, color="white", alpha=0.85))
ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]"); ax.set_aspect("equal")
ax.set_title("Stuck directly behind the obstacle")
plt.show()

## 🚁 Mini-project: the two methods, side by side

Animate the corridor-constrained trajectory and the potential-field walk on the same
scene. The QP path is smooth and was computed once; the field path is reactive and was
computed step by step. Watch how differently they behave near the pillar.

In [ ]:
grid = np.linspace(0, sum(DETOUR_TIMES), 220)
P_qp = np.array([[sample(C_avoid[a_], DETOUR_TIMES, t_) for a_ in range(3)] for t_ in grid])
P_pf = follow_field([2.0, -0.5, 1.5], [2.0, 2.5, 1.5], obstacles)
n_frames = 150

fig = plt.figure(figsize=(9.8, 4.4))

def frame(j):
    fig.clf()
    for i, (P_, name, col) in enumerate([(P_qp, "corridor QP", "C0"), (P_pf, "potential field", "C3")]):
        ax = fig.add_subplot(1, 2, i+1, projection="3d")
        k = min(int(j*len(P_)/n_frames), len(P_)-1)
        ax.plot(P_[:k+1, 0], P_[:k+1, 1], P_[:k+1, 2], color=col, lw=2.2)
        ax.plot([P_[k, 0]], [P_[k, 1]], [P_[k, 2]], "o", color=col, ms=8)
        u, v = np.mgrid[0:2*np.pi:20j, 0:np.pi:10j]
        ax.plot_surface(OBSTACLE[0] + OBS_RADIUS*np.cos(u)*np.sin(v),
                        OBSTACLE[1] + OBS_RADIUS*np.sin(u)*np.sin(v),
                        OBSTACLE[2] + OBS_RADIUS*np.cos(v), color="C3", alpha=0.3)
        ax.set_xlim(0, 3.5); ax.set_ylim(-1, 3); ax.set_zlim(0.5, 2.5)
        ax.set_title("%s\nclearance %.2f m" % (name, np.linalg.norm(P_[k] - OBSTACLE)), fontsize=9)
        ax.view_init(elev=22, azim=-68)
    return []

anim = animation.FuncAnimation(fig, frame, frames=n_frames, interval=50, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

> **🤖 Robotics connection.** Both methods are in production use, in different places.
> Potential fields (and their descendants, velocity obstacles and control barrier
> functions) run in the reactive layer, at high rate, close to the sensors. Convex
> trajectory optimisation runs in the planning layer, at a few hertz, over a map that
> something else has built. Modern systems such as the "safe flight corridor" family carve
> the free space into convex polyhedra so the QP can be told where it is allowed to be —
> which is this notebook's corridor, generalised.

**Where next.** We have a trajectory that is smooth, fast, within limits and clear of the
pillar. Notebook 10 hands it to the drone.